# News → Sentiment → Forecasts (On-Demand Batch)

Converted from SDP pipeline to run on-demand with serverless compute.

Reads bronze news tables, computes sentiment features, generates forecasts and decision signals.

| Layer | Table | Description |
|-------|-------|-------------|
| Silver | `pipelines.news_sentiment_daily` | Daily avg sentiment per symbol |
| Gold | `pipelines.news_forecast_features` | Rolling 7d/30d features joined with prices |
| Gold | `pipelines.stock_forecasts_live` | Capped + dampened momentum forecasts |
| Gold | `pipelines.decision_signals_live` | BUY/HOLD/SELL signals |

In [0]:
import logging, json, time
from datetime import datetime, timezone

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
logger = logging.getLogger("riskbricks.news_to_forecasts")

def log_step(step_name, table_name=None, row_count=None, error=None):
    """Structured step logging for job monitoring and audit trail."""
    entry = {
        "step": step_name,
        "status": "ERROR" if error else "OK",
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    if table_name:
        entry["table"] = table_name
    if row_count is not None:
        entry["rows"] = row_count
    if error:
        entry["error"] = str(error)[:300]
    logger.info(json.dumps(entry))

print("\u2705 Structured logging initialized")

In [0]:
dbutils.widgets.text('catalog', 'riskbricks')
catalog = dbutils.widgets.get('catalog').strip()

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f'USE CATALOG {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.pipelines')
print(f'Using catalog: {catalog}')

In [0]:
try:
    # GDELT events have avg_tone directly
    gdelt = (
        spark.table(f'{catalog}.bronze.historical_news_gdelt')
        .filter(F.col('symbol').isNotNull() & F.col('event_date').isNotNull())
        .groupBy('symbol', 'event_date')
        .agg(
            F.avg('avg_tone').alias('avg_sentiment'),
            F.avg('goldstein_scale').alias('avg_goldstein'),
            F.sum('num_mentions').alias('total_mentions'),
            F.sum('num_articles').alias('total_articles'),
            F.count('*').alias('event_count'),
            F.lit('gdelt').alias('primary_source')
        )
    )

    # RSS articles: count-based proxy (no tone score in RSS)
    rss = (
        spark.table(f'{catalog}.bronze.news_rss_all')
        .filter(F.col('symbol').isNotNull() & F.col('published_date').isNotNull())
        .groupBy('symbol', F.col('published_date').cast('date').alias('event_date'))
        .agg(
            F.lit(0.0).alias('avg_sentiment'),
            F.lit(0.0).alias('avg_goldstein'),
            F.count('*').alias('total_mentions'),
            F.count('*').alias('total_articles'),
            F.count('*').alias('event_count'),
            F.lit('rss').alias('primary_source')
        )
    )

    # Union and re-aggregate (a symbol may appear in both sources on same day)
    combined = gdelt.unionByName(rss)
    news_sentiment_daily = (
        combined
        .groupBy('symbol', 'event_date')
        .agg(
            F.avg('avg_sentiment').alias('daily_sentiment'),
            F.avg('avg_goldstein').alias('daily_goldstein'),
            F.sum('total_mentions').alias('daily_mentions'),
            F.sum('total_articles').alias('daily_articles'),
            F.sum('event_count').alias('daily_events')
        )
        .withColumn('computed_at', F.current_timestamp())
    )

    news_sentiment_daily.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.pipelines.news_sentiment_daily')
    cnt = spark.table(f'{catalog}.pipelines.news_sentiment_daily').count()
    log_step('news_sentiment_daily', f'{catalog}.pipelines.news_sentiment_daily', cnt)
    print(f'\u2705 news_sentiment_daily: {cnt:,} rows')

except Exception as e:
    log_step('news_sentiment_daily', error=e)
    raise

In [0]:
try:
    sentiment = spark.table(f'{catalog}.pipelines.news_sentiment_daily')
    w7 = Window.partitionBy('symbol').orderBy('event_date').rowsBetween(-6, 0)
    w30 = Window.partitionBy('symbol').orderBy('event_date').rowsBetween(-29, 0)

    sent_features = (
        sentiment
        .withColumn('event_count_7d', F.sum('daily_events').over(w7))
        .withColumn('avg_sentiment_7d', F.avg('daily_sentiment').over(w7))
        .withColumn('event_count_30d', F.sum('daily_events').over(w30))
        .withColumn('avg_sentiment_30d', F.avg('daily_sentiment').over(w30))
        .select('symbol', F.col('event_date').alias('as_of_date'),
                'event_count_7d', 'avg_sentiment_7d',
                'event_count_30d', 'avg_sentiment_30d')
    )

    # Price features from existing silver prices
    allowed = spark.table(f'{catalog}.gold.company_universe').select('symbol').distinct()
    prices = (
        spark.table(f'{catalog}.silver.stock_prices')
        .select('symbol', F.to_date('date').alias('date'), 'close')
        .join(allowed, 'symbol', 'inner')
    )

    wp = Window.partitionBy('symbol').orderBy('date')
    price_features = (
        prices
        .withColumn('return_1d', F.col('close') / F.lag('close').over(wp) - 1.0)
        .withColumn('return_5d', F.col('close') / F.lag('close', 5).over(wp) - 1.0)
        .withColumn('return_20d', F.col('close') / F.lag('close', 20).over(wp) - 1.0)
        .withColumn('volatility_20d', F.stddev('return_1d').over(wp.rowsBetween(-19, 0)))
        .select(F.col('symbol'), F.col('date').alias('as_of_date'),
                F.col('close').alias('last_close'),
                'return_5d', 'return_20d', 'volatility_20d')
    )

    # Join price + sentiment
    news_forecast_features = (
        price_features
        .join(sent_features, ['symbol', 'as_of_date'], 'left')
        .fillna({
            'event_count_7d': 0, 'avg_sentiment_7d': 0.0,
            'event_count_30d': 0, 'avg_sentiment_30d': 0.0
        })
        .withColumn('computed_at', F.current_timestamp())
    )

    news_forecast_features.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.pipelines.news_forecast_features')
    cnt = spark.table(f'{catalog}.pipelines.news_forecast_features').count()
    log_step('news_forecast_features', f'{catalog}.pipelines.news_forecast_features', cnt)
    print(f'\u2705 news_forecast_features: {cnt:,} rows')

except Exception as e:
    log_step('news_forecast_features', error=e)
    raise

In [0]:
try:
    features = spark.table(f'{catalog}.pipelines.news_forecast_features')
    w_latest = Window.partitionBy('symbol').orderBy(F.col('as_of_date').desc())
    latest = features.withColumn('rn', F.row_number().over(w_latest)).filter(F.col('rn') == 1).drop('rn')

    # Forecast formula: 30% sentiment + 70% momentum
    sent7 = F.col('avg_sentiment_7d') / 10.0
    sent30 = F.col('avg_sentiment_30d') / 10.0
    ret5 = F.coalesce(F.col('return_5d'), F.lit(0.0))
    ret20 = F.coalesce(F.col('return_20d'), F.col('return_5d'), F.lit(0.0))
    vol = F.coalesce(F.col('volatility_20d'), F.lit(0.02))

    # 1-DAY FORECAST: Capped momentum + volatility dampening + mean reversion
    capped_ret5 = F.greatest(F.least(ret5, F.lit(0.05)), F.lit(-0.05))
    daily_mom_1d = capped_ret5 / F.lit(5.0)
    vol_dampen = F.greatest(F.lit(1.0) - (vol * F.lit(10.0)), F.lit(0.3))
    reversion_1d = F.when(F.abs(ret5) > 0.10, -0.3 * ret5).otherwise(F.lit(0.0))
    new_mom_1d = daily_mom_1d * vol_dampen + reversion_1d
    pred_1d = (F.lit(0.30) * sent7) + (F.lit(0.70) * new_mom_1d)

    # 15-DAY FORECAST
    capped_ret20 = F.greatest(F.least(ret20, F.lit(0.15)), F.lit(-0.15))
    daily_mom_15d = capped_ret20 / F.lit(20.0) * F.lit(15.0)
    reversion_15d = F.when(F.abs(ret20) > 0.15, -0.2 * ret20).otherwise(F.lit(0.0))
    new_mom_15d = daily_mom_15d * vol_dampen + reversion_15d
    pred_15d = (F.lit(0.30) * sent30) + (F.lit(0.70) * new_mom_15d)

    base = latest.withColumn('forecast_date', F.col('as_of_date'))

    fc_1d = (
        base
        .withColumn('horizon_days', F.lit(1))
        .withColumn('predicted_price', F.col('last_close') * (1.0 + pred_1d))
        .withColumn('predicted_direction', F.when(pred_1d >= 0, 'up').otherwise('down'))
        .withColumn('confidence_band_low', F.col('last_close') * (1.0 - vol))
        .withColumn('confidence_band_high', F.col('last_close') * (1.0 + vol))
    )

    fc_15d = (
        base
        .withColumn('horizon_days', F.lit(15))
        .withColumn('predicted_price', F.col('last_close') * (1.0 + pred_15d))
        .withColumn('predicted_direction', F.when(pred_15d >= 0, 'up').otherwise('down'))
        .withColumn('confidence_band_low', F.col('last_close') * (1.0 - vol * 2.0))
        .withColumn('confidence_band_high', F.col('last_close') * (1.0 + vol * 2.0))
    )

    stock_forecasts_live = (
        fc_1d.unionByName(fc_15d)
        .select(
            'symbol', 'forecast_date', 'horizon_days',
            'predicted_price', 'predicted_direction',
            'confidence_band_low', 'confidence_band_high',
            'last_close', 'avg_sentiment_7d', 'avg_sentiment_30d',
            'event_count_7d', 'event_count_30d',
            'return_5d', 'return_20d', 'volatility_20d'
        )
        .withColumn('computed_at', F.current_timestamp())
    )

    stock_forecasts_live.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.pipelines.stock_forecasts_live')
    cnt = spark.table(f'{catalog}.pipelines.stock_forecasts_live').count()
    log_step('stock_forecasts_live', f'{catalog}.pipelines.stock_forecasts_live', cnt)
    print(f'\u2705 stock_forecasts_live: {cnt:,} rows')

except Exception as e:
    log_step('stock_forecasts_live', error=e)
    raise

In [0]:
try:
    forecasts = spark.table(f'{catalog}.pipelines.stock_forecasts_live')
    universe = spark.table(f'{catalog}.gold.company_universe').select('symbol', 'beta')

    decision_signals_live = (
        forecasts
        .join(universe, 'symbol', 'left')
        .withColumn(
            'pct_change',
            (F.col('predicted_price') - F.col('last_close')) / F.col('last_close')
        )
        .withColumn(
            'signal',
            F.when((F.col('predicted_direction') == 'up') & (F.abs(F.col('pct_change')) > 0.03), 'BUY')
             .when((F.col('predicted_direction') == 'down') & (F.abs(F.col('pct_change')) > 0.03), 'SELL')
             .otherwise('HOLD')
        )
        .select(
            'symbol',
            F.col('forecast_date').alias('as_of_date'),
            F.date_add('forecast_date', F.col('horizon_days')).alias('target_date'),
            'signal',
            F.round(F.col('pct_change') * 100, 2).alias('score'),
            F.round(F.col('pct_change'), 4).alias('expected_return'),
            'horizon_days',
            'volatility_20d',
            F.col('beta').alias('beta_1y'),
            'event_count_30d',
            F.when(F.col('event_count_30d') > 0, 1).otherwise(0).alias('news_source_count'),
            F.current_timestamp().alias('pipeline_timestamp')
        )
    )

    decision_signals_live.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.pipelines.decision_signals_live')
    cnt = spark.table(f'{catalog}.pipelines.decision_signals_live').count()
    log_step('decision_signals_live', f'{catalog}.pipelines.decision_signals_live', cnt)
    print(f'\u2705 decision_signals_live: {cnt:,} rows')

except Exception as e:
    log_step('decision_signals_live', error=e)
    raise

In [0]:
print('=' * 60)
print('\U0001f4f0 NEWS \u2192 FORECASTS REFRESH COMPLETE')
print('=' * 60)
summary_tables = ['news_sentiment_daily', 'news_forecast_features', 'stock_forecasts_live', 'decision_signals_live']
for table in summary_tables:
    cnt = spark.table(f'{catalog}.pipelines.{table}').count()
    log_step('summary', f'{catalog}.pipelines.{table}', cnt)
    print(f'  \u2705 pipelines.{table}: {cnt:,} rows')
print('=' * 60)
logger.info('Pipeline completed successfully')

In [0]:
# ── Sync pipelines.*_live → gold.* so the UC agent tools see fresh data ──
from pyspark.sql import functions as F

catalog = dbutils.widgets.get('catalog').strip()
print(f"\n{'=' * 60}")
print("SYNCING PIPELINE TABLES → GOLD LAYER")
print(f"{'=' * 60}")

# ── 1. decision_signals_live → gold.decision_signals ──
try:
    live_ds = spark.table(f'{catalog}.pipelines.decision_signals_live')
    gold_ds = (
        live_ds
        .select(
            'symbol',
            'as_of_date',
            'target_date',
            'signal',
            'score',
            'expected_return',
            F.lit(1).cast('bigint').alias('model_count'),
            F.col('volatility_20d').alias('vol_20d'),
            'beta_1y',
            F.col('event_count_30d').alias('news_doc_count'),
            F.col('news_source_count').cast('bigint'),
            F.lit(0).cast('bigint').alias('earnings_count'),
            F.lit(0).cast('bigint').alias('analyst_count'),
            F.lit(None).cast('double').alias('options_iv_skew'),
            F.lit(None).cast('double').alias('short_ratio'),
            F.col('pipeline_timestamp').alias('ingestion_timestamp'),
        )
    )
    gold_ds.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.gold.decision_signals')
    cnt = spark.table(f'{catalog}.gold.decision_signals').count()
    latest = spark.sql(f"SELECT MAX(as_of_date) FROM {catalog}.gold.decision_signals").first()[0]
    log_step('sync_gold', f'{catalog}.gold.decision_signals', cnt)
    print(f'  ✅ gold.decision_signals: {cnt:,} rows, latest = {latest}')
except Exception as e:
    log_step('sync_gold', f'{catalog}.gold.decision_signals', error=e)
    print(f'  ❌ gold.decision_signals: {e}')

# ── 2. stock_forecasts_live → gold.stock_forecasts ──
try:
    live_sf = spark.table(f'{catalog}.pipelines.stock_forecasts_live')
    gold_sf = (
        live_sf
        .select(
            'symbol',
            F.col('forecast_date').alias('as_of_date'),
            'last_close',
            'return_5d',
            'return_20d',
            'volatility_20d',
            'event_count_7d',
            F.round('avg_sentiment_7d', 1).alias('avg_sentiment_7d'),
            'event_count_30d',
            F.round('avg_sentiment_30d', 1).alias('avg_sentiment_30d'),
            F.col('event_count_30d').alias('evidence_count_30d'),
            'forecast_date',
            'horizon_days',
            'predicted_price',
            'predicted_direction',
            'confidence_band_low',
            'confidence_band_high',
            F.lit(None).cast('array<string>').alias('top_factors'),
            F.lit(None).cast('string').alias('feature_snapshot'),
            'computed_at',
        )
    )
    gold_sf.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.gold.stock_forecasts')
    cnt = spark.table(f'{catalog}.gold.stock_forecasts').count()
    latest = spark.sql(f"SELECT MAX(as_of_date) FROM {catalog}.gold.stock_forecasts").first()[0]
    log_step('sync_gold', f'{catalog}.gold.stock_forecasts', cnt)
    print(f'  ✅ gold.stock_forecasts: {cnt:,} rows, latest = {latest}')
except Exception as e:
    log_step('sync_gold', f'{catalog}.gold.stock_forecasts', error=e)
    print(f'  ❌ gold.stock_forecasts: {e}')

# ── 3. news_forecast_features → silver.forecast_features_daily ──
# The ML feature pipeline (SDP) reads from silver.forecast_features_daily
# to assemble gold.ml_prediction_features. Keep it synced from the batch pipeline.
try:
    nff = spark.table(f'{catalog}.pipelines.news_forecast_features')
    ffd = (
        nff
        .select(
            'symbol',
            'as_of_date',
            'last_close',
            'return_5d',
            'return_20d',
            'volatility_20d',
            'event_count_7d',
            'avg_sentiment_7d',
            'event_count_30d',
            'avg_sentiment_30d',
            F.col('event_count_30d').alias('evidence_count_30d'),
        )
    )
    ffd.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{catalog}.silver.forecast_features_daily')
    cnt = spark.table(f'{catalog}.silver.forecast_features_daily').count()
    latest = spark.sql(f"SELECT MAX(as_of_date) FROM {catalog}.silver.forecast_features_daily").first()[0]
    log_step('sync_silver', f'{catalog}.silver.forecast_features_daily', cnt)
    print(f'  ✅ silver.forecast_features_daily: {cnt:,} rows, latest = {latest}')
except Exception as e:
    log_step('sync_silver', f'{catalog}.silver.forecast_features_daily', error=e)
    print(f'  ❌ silver.forecast_features_daily: {e}')

print(f"{'=' * 60}")